# # YOLO Annotatoions Generator
This notebook is used to generate YOLO annotations by using a pre-trained YOLO model.

## Imports

In [1]:
import sys
import cv2
from pathlib import Path
import json
import shutil

In [2]:
# Set execution root in the project root
sys.path.insert(0, str(Path.cwd().parent.parent.parent))

In [3]:
from src.utils.config.config import Config
from src.detector.yolo_detector import YOLODetector

## Config

In [4]:
config_loader = Config()
cfg = config_loader.load_config()

yolo_model_type = cfg.paths.models / config_loader.get("model", "yolo.pretrained")
yolo_imgsz = config_loader.get("model", "yolo.imgsz")

raw_dataset = cfg.project_root / config_loader.get("paths", "dataset.raw") / "CafeV1"

output_dataset = raw_dataset / "yolo"

In [5]:
ALLOWED_CLASSES = [
    0,   # person
    63,  # laptop
    67,  # cell phone
    73   # book
]

## Init YOLO

In [6]:
detector = YOLODetector(yolo_model_type, cfg.project_root)

YOLOv8m summary: 169 layers, 25,902,640 parameters, 0 gradients, 79.3 GFLOPs


## Functions

### load_existing_json

In [7]:
def load_existing_json(json_path):
    if json_path.exists():
        with open(json_path, "r") as f:
            return json.load(f)
    # Empty data
    return None

### save_json

In [ ]:
def save_json(data, json_path):
    with open(json_path, "w") as f:
        json.dump(data, f, indent=4)

### process_image

In [13]:
def process_image(image_path, image_id, annotation_id):
    # Do inference on the frame
    results = detector.model(
        source=image_path,
        verbose=False,
        imgsz = yolo_imgsz, # 640, 960, 1280,
        classes = ALLOWED_CLASSES,
        conf=0.15
    )[0]

    # YOLO built-in visualizer (used to verify)
    results.show()

    detections = []
    if results.boxes is None:
        return detections

    boxes = results.boxes.xyxy.cpu().numpy()
    classes = results.boxes.cls.cpu().numpy()

    for i in range(len(boxes)):
        bbox = boxes[i]
        cls = int(classes[i])
        x1, y1, x2, y2 = bbox.tolist() 
        w = x2 - x1
        h = y2 - y1
        
        # Format into JSON
        detections.append({
            "id": annotation_id,
            "image_id": image_id,
            "category_id": cls,
            "bbox": [x1, y1, w, h],
            "area": w * h,
            "iscrowd": 0,
            "attributes": {}
        })
        annotation_id+=1
        
    return detections, annotation_id

### process_clip

In [ ]:
def process_clip(clip_path):
    clip_name = clip_path.name # 0
    
    images_path = clip_path / "images" 
    
    output_path = output_dataset / "clips" / clip_name
    output_images_path = output_path / "images"  
    output_json_path = output_path / "anns.json"
    
    if not images_path.exists():
        print(f"Couldn't process clip {clip_name}")
        return

    # Get and sort frames
    image_files = (p for p in images_path.iterdir()) # list of frames
    image_files = sorted(image_files, key=lambda x: int(x.stem.split("_")[1]))

    # Ensure path exists
    output_images_path.mkdir(parents=True, exist_ok=True)

    json_data = load_existing_json(output_json_path)
    
    # Create default JSON if not found
    if json_data == None:
        json_data = {
            "images": [],
            "annotations": [],
            "categories": [
                {"id": 0, "name": "person"},
                {"id": 63, "name": "laptop"},
                {"id": 67, "name": "cell phone"},
                {"id": 73, "name": "book"},
                {"id": 41, "name": "cup"}
            ],
            "hoi_annotations": []
        }
        
    annotation_id = 0
    
    processed_frames = {
        f["id"] 
        for f in (json_data.get("images") or [])
    }
    
    for image_id, image_file in enumerate(image_files):
        # Skip already processed frames
        if image_id in processed_frames:
            continue 
        
        image_path = images_path / image_file
        
        # Add image entry
        json_data["images"].append({
            "id": image_id,
            "file_name": image_file.name,
            "width": 1920,
            "height": 1080
        })

        detections, latest_annotation_id = process_image(image_path, image_id, annotation_id)
        annotation_id = latest_annotation_id

        # Add detections data for current frame
        json_data["annotations"].extend(detections)
        
        # Save frame image
        shutil.copy2(image_path, output_images_path)
        # break # temp - used to validate one frame

    save_json(json_data, output_json_path)
    # print(f"Saved anns for clip {clip_name} at {output_json_path}")

## Run

In [ ]:
def main():
    clips_path = raw_dataset / "Clips"
    clip_dirs = sorted(p for p in clips_path.iterdir() if p.is_dir())

    for clip_path in clip_dirs:
        process_clip(clip_path)
        # return # temp - used to validate one clip

main()